In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [32]:
X1 = np.array([-1.0, -1.0, 1.0, 1.0])
X2 = np.array([-1.0, 1.0, -1.0, 1.0])
Y  = np.array([-2.2, -1.8, 1.8, 2.2])
m = len(Y)

w1_0 = 0.0
w2_0 = 0.03333
b_0  = 0.0
alpha = 0.1
lam   = 0.5

In [33]:
df = pd.DataFrame({'Obs': [1, 2, 3, 4], 'X1': X1, 'X2': X2, 'Y': Y})
print("=== Dataset ===")
print(df.to_string(index=False))

=== Dataset ===
 Obs   X1   X2    Y
   1 -1.0 -1.0 -2.2
   2 -1.0  1.0 -1.8
   3  1.0 -1.0  1.8
   4  1.0  1.0  2.2


In [34]:
def lasso_gradient_descent(X1, X2, Y, w1_init, w2_init, b_init, alpha, lam, num_iters=3):
    w1, w2, b = w1_init, w2_init, b_init
    history = []

    for i in range(num_iters):
        y_pred = w1 * X1 + w2 * X2 + b
        err = y_pred - Y

        subgrad1 = np.sign(w1) if w1 != 0 else 0.0
        subgrad2 = np.sign(w2) if w2 != 0 else 0.0

        dw1 = (1/m) * np.sum(err * X1) + (lam / m) * subgrad1
        dw2 = (1/m) * np.sum(err * X2) + (lam / m) * subgrad2
        db  = (1/m) * np.sum(err)

        w1_next = w1 - alpha * dw1
        w2_next = w2 - alpha * dw2
        b_next  = b - alpha * db

        if np.sign(w2) != np.sign(w2_next) and w2 != 0:
            w2_next = 0.0

        w1, w2, b = w1_next, w2_next, b_next
        mse = (1/(2*m)) * np.sum(err**2)
        history.append((i+1, w1, w2, b, dw1, dw2, mse))

    return w1, w2, b, pd.DataFrame(history, columns=['Iter', 'w1', 'w2', 'b', 'dw1', 'dw2', 'MSE'])

In [35]:
def ridge_gradient_descent(X1, X2, Y, w1_init, w2_init, b_init, alpha, lam, num_iters=3):
    w1, w2, b = w1_init, w2_init, b_init
    history = []

    for i in range(num_iters):
        y_pred = w1 * X1 + w2 * X2 + b
        err = y_pred - Y

        dw1 = (1/m) * np.sum(err * X1) + (lam / m) * w1
        dw2 = (1/m) * np.sum(err * X2) + (lam / m) * w2
        db  = (1/m) * np.sum(err)

        w1 -= alpha * dw1
        w2 -= alpha * dw2
        b  -= alpha * db

        mse = (1/(2*m)) * np.sum(err**2)
        history.append((i+1, w1, w2, b, dw1, dw2, mse))

    return w1, w2, b, pd.DataFrame(history, columns=['Iter', 'w1', 'w2', 'b', 'dw1', 'dw2', 'MSE'])

In [36]:
_, _, _, ridge_history = ridge_gradient_descent(X1, X2, Y, w1_0, w2_0, b_0, alpha, lam, num_iters=3)
_, _, _, lasso_history = lasso_gradient_descent(X1, X2, Y, w1_0, w2_0, b_0, alpha, lam, num_iters=3)

In [37]:
pd.options.display.float_format = '{:,.2f}'.format

print("=== Ridge Regression (Detailed History) ===")
display(ridge_history)

print("\n=== Lasso Regression (Detailed History) ===")
display(lasso_history)

=== Ridge Regression (Detailed History) ===


,Iter,w1,w2,b,dw1,dw2,MSE
0,1,0.20,0.05,0.00,-2.00,-0.16,2.01
1,2,0.38,0.06,0.00,-1.78,-0.14,1.63
2,3,0.54,0.08,0.00,-1.58,-0.13,1.33



=== Lasso Regression (Detailed History) ===


,Iter,w1,w2,b,dw1,dw2,MSE
0,1,0.20,0.04,0.00,-2.00,-0.04,2.01
1,2,0.37,0.04,0.00,-1.68,-0.04,1.63
2,3,0.52,0.04,0.00,-1.51,-0.03,1.35
